# Lung Segmentation — Preprocessing Pipeline v3
### 4-Class Unified Mask Schema
| Class | Value | Source |
|-------|-------|--------|
| Background | 0 | Both datasets |
| Lung | 1 | COVID LungMasks (binary PNG) + LUNA seg .mhd (values 3,4,5 → 1) |
| Nodule | 2 | LUNA annotations.csv drawn as 3D sphere |
| Infection (GGO / Consolidation) | 3 | COVID InfectionMasks (binary PNG) |

**LUNA16 seg mask values:**  
`0` = background, `3` = right lung, `4` = left lung, `5` = trachea/combined  
All non-zero values are mapped to Class 1 (lung).

In [ ]:
!pip install SimpleITK -q

In [ ]:
import os
import glob
import numpy as np
import cv2
import pandas as pd
import SimpleITK as sitk
from tqdm import tqdm
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────

# COVID dataset (unchanged)
COVID_BASE          = '/kaggle/input/datasets/nguyentienda32143/covid-19-ct-lung-and-infection-segmentation/data/data'
COVID_IMG_DIR       = os.path.join(COVID_BASE, 'Covid-19-2')
COVID_INF_MASK_DIR  = os.path.join(COVID_BASE, 'InfectionMasks')
COVID_LUNG_MASK_DIR = os.path.join(COVID_BASE, 'LungMasks')

# LUNA16 — now TWO separate directories:
# Raw CT volumes (newly downloaded subset0)
LUNA_CT_DIR    = '/kaggle/input/datasets/swarajjamkar/luna16-subset0-raw-ct/luna16_subset0/subset0'
# Lung seg masks (the seg-lungs-LUNA16 dataset you already had)
# These are the files with unique values [0, 3, 4, 5]
LUNA_SEG_DIR   = '/kaggle/input/datasets/fanbyprinciple/luna-lung-cancer-dataset/seg-lungs-LUNA16/seg-lungs-LUNA16'
LUNA_ANNOTATIONS = '/kaggle/input/datasets/fanbyprinciple/luna-lung-cancer-dataset/annotations.csv'

OUTPUT_IMG_DIR  = '/kaggle/working/processed_data/images'
OUTPUT_MASK_DIR = '/kaggle/working/processed_data/masks'
os.makedirs(OUTPUT_IMG_DIR, exist_ok=True)
os.makedirs(OUTPUT_MASK_DIR, exist_ok=True)

# ─────────────────────────────────────────────
# CLASS CONSTANTS
# ─────────────────────────────────────────────
CLASS_BACKGROUND = 0
CLASS_LUNG       = 1
CLASS_NODULE     = 2
CLASS_INFECTION  = 3
NUM_CLASSES      = 4

TARGET_SIZE = (256, 256)

# Every Nth non-nodule LUNA slice saved as background example
BACKGROUND_SLICE_EVERY_N = 10

print('Paths configured.')

# ── Verify all directories exist before starting ──
paths_to_check = {
    'COVID images'       : COVID_IMG_DIR,
    'COVID inf masks'    : COVID_INF_MASK_DIR,
    'COVID lung masks'   : COVID_LUNG_MASK_DIR,
    'LUNA CT volumes'    : LUNA_CT_DIR,
    'LUNA seg masks'     : LUNA_SEG_DIR,
    'LUNA annotations'   : LUNA_ANNOTATIONS,
}
all_ok = True
for name, path in paths_to_check.items():
    exists = os.path.exists(path)
    status = 'OK' if exists else 'MISSING'
    print(f'  [{status}]  {name}: {path}')
    if not exists:
        all_ok = False

if all_ok:
    print('\nAll paths found. Ready to run.')
else:
    print('\nFix missing paths before continuing.')

## Shared Utility Functions

In [ ]:
from numpy import uint8


def window_and_normalize(image_hu, wl=-600, ww=1500):
    """
    CT lung windowing + normalize to [0, 1] float32.
    WL=-600, WW=1500  →  HU range [-1350, 150]
    Confirmed correct for our volumes (HU min -3024, max 2103).
    """
    hu_min = wl - (ww / 2)
    hu_max = wl + (ww / 2)
    return np.clip(
        (image_hu.astype(np.float32) - hu_min) / (hu_max - hu_min),
        0.0, 1.0
    )


def normalize_uint8_image(image_uint8):
    """Normalize a uint8 PNG (0-255) to [0, 1] float32."""
    return (image_uint8.astype(np.float32) / 255.0)


def binarize_mask(mask, threshold=127):
    """Convert grayscale mask to clean binary (0 or 1)."""
    return (mask > threshold).astype(np.uint8)


def world_to_voxel(world_coord, origin, spacing):
    """Convert physical mm coordinates to voxel indices. All (x,y,z) ordered."""
    return np.round(np.abs(world_coord - origin) / spacing).astype(int)


def save_pair(img_float32, mask_uint8, name):
    np.save(os.path.join(OUTPUT_IMG_DIR,  name), img_float32)
    np.save(os.path.join(OUTPUT_MASK_DIR, name), mask_uint8)


print('Utility functions ready.')


## Dataset 1 — COVID-19
Both mask types are binary PNGs (white = region, black = background).  
Mask build order: Background → Lung (class 1) → Infection (class 3, overwrites lung at overlap).

In [ ]:
def process_covid_slices():
    print('Processing COVID-19 dataset (full dataset, no slice limit)...')

    inf_mask_paths = sorted(glob.glob(os.path.join(COVID_INF_MASK_DIR, '*.*')))
    saved, skipped = 0, 0

    for inf_path in tqdm(inf_mask_paths):
        filename  = os.path.basename(inf_path)
        stem      = os.path.splitext(filename)[0]
        img_path  = os.path.join(COVID_IMG_DIR,       filename)
        lung_path = os.path.join(COVID_LUNG_MASK_DIR, filename)

        img       = cv2.imread(img_path,  cv2.IMREAD_GRAYSCALE)
        inf_mask  = cv2.imread(inf_path,  cv2.IMREAD_GRAYSCALE)
        lung_mask = cv2.imread(lung_path, cv2.IMREAD_GRAYSCALE) if os.path.exists(lung_path) else None

        if img is None or inf_mask is None:
            skipped += 1
            continue

        img      = cv2.resize(img,      TARGET_SIZE, interpolation=cv2.INTER_LINEAR)
        inf_mask = cv2.resize(inf_mask, TARGET_SIZE, interpolation=cv2.INTER_NEAREST)
        if lung_mask is not None:
            lung_mask = cv2.resize(lung_mask, TARGET_SIZE, interpolation=cv2.INTER_NEAREST)

        # Normalize image to [0,1] float32 — same scale as LUNA HU windowing
        img_normalized = normalize_uint8_image(img)

        # Build unified mask
        unified_mask = np.zeros(TARGET_SIZE, dtype=np.uint8)
        if lung_mask is not None:
            unified_mask[binarize_mask(lung_mask) == 1] = CLASS_LUNG
        unified_mask[binarize_mask(inf_mask) == 1] = CLASS_INFECTION

        save_pair(img_normalized, unified_mask, f'covid_{stem}.npy')
        saved += 1

    print(f'COVID done — saved: {saved:,}  |  skipped: {skipped}')


process_covid_slices()

## Dataset 2 — LUNA16 (subset0)

**Two separate directories:**
- Raw CT `.mhd` files → `LUNA_CT_DIR` (newly downloaded, HU values)
- Lung seg `.mhd` files → `LUNA_SEG_DIR` (existing dataset, values 0/3/4/5)

**Seg mask value mapping:**  
`0` → background, `3` → right lung → class 1, `4` → left lung → class 1, `5` → trachea → class 1  
All non-zero values become Class 1 (lung).

**Slice selection:**
- Nodule slices → always saved
- Non-nodule slices → every 10th saved as background examples

In [ ]:
def build_ct_seg_pairs(ct_dir, seg_dir):
    """
    Pairs raw CT .mhd files with their lung seg .mhd files by matching UIDs.

    LUNA16 naming: both CT and seg files share the same UID filename.
    They just live in different directories.

    Returns:
        list of (ct_path, seg_path) tuples
        Only includes pairs where BOTH files exist.
    """
    ct_files  = sorted(glob.glob(os.path.join(ct_dir,  '*.mhd')))
    seg_files = sorted(glob.glob(os.path.join(seg_dir, '*.mhd')))

    # Build UID -> path lookup for seg files
    seg_lookup = {os.path.basename(p).replace('.mhd', ''): p for p in seg_files}

    pairs     = []
    unmatched = []

    for ct_path in ct_files:
        uid      = os.path.basename(ct_path).replace('.mhd', '')
        seg_path = seg_lookup.get(uid, None)
        if seg_path:
            pairs.append((ct_path, seg_path))
        else:
            unmatched.append(uid)

    print(f'CT volumes found     : {len(ct_files)}')
    print(f'Seg masks found      : {len(seg_files)}')
    print(f'Matched pairs        : {len(pairs)}')
    if unmatched:
        print(f'Unmatched CT files   : {len(unmatched)}')
        print(f'  (these will be skipped — no seg mask available)')

    return pairs


ct_seg_pairs = build_ct_seg_pairs(LUNA_CT_DIR, LUNA_SEG_DIR)
print(f'\nReady to process {len(ct_seg_pairs)} paired volumes.')

In [ ]:
def process_luna16(ct_seg_pairs, n=BACKGROUND_SLICE_EVERY_N):
    print(f'Processing LUNA16 subset0 — {len(ct_seg_pairs)} volumes...')
    print(f'Background slice sampling: every {n}th non-nodule slice\n')

    try:
        annotations = pd.read_csv(LUNA_ANNOTATIONS)
        print(f'Loaded annotations.csv — {len(annotations)} nodule records.')
    except FileNotFoundError:
        print(f'ERROR: annotations.csv not found at {LUNA_ANNOTATIONS}')
        return

    total_nodule_slices = 0
    total_bg_slices     = 0

    for ct_path, seg_path in tqdm(ct_seg_pairs):
        patient_id = os.path.basename(ct_path).replace('.mhd', '')

        # ── Load raw CT volume ──
        itk_ct    = sitk.ReadImage(ct_path)
        img_array = sitk.GetArrayFromImage(itk_ct)   # (Z, Y, X), int16, HU values
        origin    = np.array(itk_ct.GetOrigin())     # (x, y, z) mm
        spacing   = np.array(itk_ct.GetSpacing())    # (x, y, z) mm/voxel
        n_slices  = img_array.shape[0]

        # ── Load lung seg mask ──
        # Values are 0 (background), 3 (right lung), 4 (left lung), 5 (trachea)
        # All non-zero → Class 1 (lung)
        itk_seg       = sitk.ReadImage(seg_path)
        seg_array     = sitk.GetArrayFromImage(itk_seg)              # (Z, Y, X), int16
        lung_array_3d = (seg_array > 0).astype(np.uint8)            # binary: 1 = any lung tissue

        # ── Window entire CT volume once (faster than per-slice) ──
        img_windowed = window_and_normalize(img_array)               # (Z, Y, X), float32 [0,1]

        # ── Build 3D nodule mask from annotations.csv ──
        nodule_mask_3d  = np.zeros_like(img_array, dtype=np.uint8)
        patient_nodules = annotations[annotations['seriesuid'] == patient_id]
        nodule_z_set    = set()

        for _, nodule in patient_nodules.iterrows():
            world_coord = np.array([nodule['coordX'], nodule['coordY'], nodule['coordZ']])
            diameter_mm = nodule['diameter_mm']

            v_x, v_y, v_z = world_to_voxel(world_coord, origin, spacing)
            # spacing[0] is x-spacing (in-plane), used to convert mm radius to pixels
            radius_px     = max(1, int((diameter_mm / 2.0) / spacing[0]))

            z_min = max(0, v_z - radius_px);  z_max = min(n_slices,           v_z + radius_px + 1)
            y_min = max(0, v_y - radius_px);  y_max = min(img_array.shape[1], v_y + radius_px + 1)
            x_min = max(0, v_x - radius_px);  x_max = min(img_array.shape[2], v_x + radius_px + 1)

            # Vectorized sphere drawing
            zz, yy, xx = np.ogrid[z_min:z_max, y_min:y_max, x_min:x_max]
            sphere = ((zz-v_z)**2 + (yy-v_y)**2 + (xx-v_x)**2) <= radius_px**2
            nodule_mask_3d[z_min:z_max, y_min:y_max, x_min:x_max][sphere] = CLASS_NODULE

            for z in range(z_min, z_max):
                nodule_z_set.add(z)

        # ── Determine which slices to save ──
        slices_to_save = {}   # z -> 'nodule' or 'background'
        for z in range(n_slices):
            if z in nodule_z_set:
                slices_to_save[z] = 'nodule'
            elif z % n == 0:
                slices_to_save[z] = 'background'

        # ── Extract and save 2D slices ──
        for z_slice, slice_type in slices_to_save.items():

            img_resized = cv2.resize(
                img_windowed[z_slice],
                TARGET_SIZE,
                interpolation=cv2.INTER_LINEAR
            )

            # Build unified 4-class mask for this slice
            unified_mask = np.zeros(img_array.shape[1:], dtype=np.uint8)   # (Y, X)

            # Layer 1: Lung (class 1) — all non-zero seg values
            unified_mask[lung_array_3d[z_slice] > 0] = CLASS_LUNG

            # Layer 2: Nodule (class 2) — overwrites lung where nodule exists
            unified_mask[nodule_mask_3d[z_slice] == CLASS_NODULE] = CLASS_NODULE

            mask_resized = cv2.resize(
                unified_mask,
                TARGET_SIZE,
                interpolation=cv2.INTER_NEAREST   # NEAREST is mandatory for masks
            )

            save_pair(img_resized, mask_resized, f'luna_{patient_id}_z{z_slice:04d}.npy')

            if slice_type == 'nodule':
                total_nodule_slices += 1
            else:
                total_bg_slices += 1

    print(f'\nLUNA16 done.')
    print(f'  Nodule slices     : {total_nodule_slices:,}')
    print(f'  Background slices : {total_bg_slices:,}  (every {n}th non-nodule slice)')
    print(f'  Total LUNA slices : {total_nodule_slices + total_bg_slices:,}')


process_luna16(ct_seg_pairs)

## Sanity Check — Visualize Random Samples
Confirms images are [0,1] float32, masks have only values 0-3, and class colors look correct.

In [ ]:
CLASS_COLORS = np.array([
    [0,   0,   0  ],   # 0 background  — black
    [0,   200, 0  ],   # 1 lung        — green
    [220, 50,  50 ],   # 2 nodule      — red
    [255, 220, 0  ],   # 3 infection   — yellow
], dtype=np.uint8)
CLASS_NAMES = {0: 'BG', 1: 'Lung', 2: 'Nodule', 3: 'Infection'}


def colorize_mask(mask):
    rgb = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for cls_id, color in enumerate(CLASS_COLORS):
        rgb[mask == cls_id] = color
    return rgb


def visualize_samples(n_each=3, seed=42):
    all_imgs = sorted(glob.glob(os.path.join(OUTPUT_IMG_DIR, '*.npy')))
    if not all_imgs:
        print('No files found. Run both pipelines first.'); return

    covid_imgs = [p for p in all_imgs if os.path.basename(p).startswith('covid_')]
    luna_imgs  = [p for p in all_imgs if os.path.basename(p).startswith('luna_')]
    # For LUNA, prefer slices that have nodules
    luna_nodule_imgs = []
    for p in luna_imgs:
        mask = np.load(p.replace(OUTPUT_IMG_DIR, OUTPUT_MASK_DIR))
        if CLASS_NODULE in mask:
            luna_nodule_imgs.append(p)

    np.random.seed(seed)
    chosen = []
    if covid_imgs:        chosen += list(np.random.choice(covid_imgs,        min(n_each, len(covid_imgs)),        replace=False))
    if luna_nodule_imgs:  chosen += list(np.random.choice(luna_nodule_imgs,  min(n_each, len(luna_nodule_imgs)),  replace=False))
    elif luna_imgs:       chosen += list(np.random.choice(luna_imgs,         min(n_each, len(luna_imgs)),         replace=False))

    fig, axes = plt.subplots(len(chosen), 3, figsize=(12, len(chosen) * 4))
    if len(chosen) == 1: axes = [axes]

    for row, img_path in enumerate(chosen):
        mask_path = img_path.replace(OUTPUT_IMG_DIR, OUTPUT_MASK_DIR)
        img  = np.load(img_path)
        mask = np.load(mask_path)
        name = os.path.basename(img_path)

        # Assertions — catch any preprocessing bugs immediately
        assert img.dtype == np.float32,               f'Image dtype: {img.dtype}'
        assert 0.0 <= img.min() and img.max() <= 1.0, f'Image range: [{img.min()}, {img.max()}]'
        assert set(np.unique(mask)).issubset({0,1,2,3}), f'Bad mask values: {np.unique(mask)}'

        mask_color = colorize_mask(mask)
        overlay    = np.stack([img]*3, axis=-1)
        blend      = np.clip(overlay * 0.6 + mask_color / 255.0 * 0.4, 0, 1)
        classes    = ', '.join(CLASS_NAMES[c] for c in np.unique(mask))

        axes[row][0].imshow(img, cmap='gray', vmin=0, vmax=1)
        axes[row][0].set_title(f'{name}\nClasses: {classes}', fontsize=7)
        axes[row][0].axis('off')
        axes[row][1].imshow(mask_color)
        axes[row][1].set_title('Mask', fontsize=8)
        axes[row][1].axis('off')
        axes[row][2].imshow(blend)
        axes[row][2].set_title('Overlay', fontsize=8)
        axes[row][2].axis('off')

    plt.tight_layout()
    out_path = '/kaggle/working/sample_verification.png'
    plt.savefig(out_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'All assertions passed. Saved to {out_path}')


visualize_samples()

## Dataset Statistics + Class Weights
Run before training. The printed weights go directly into your `CrossEntropyLoss`.

In [ ]:
def compute_dataset_stats():
    all_masks = sorted(glob.glob(os.path.join(OUTPUT_MASK_DIR, '*.npy')))
    if not all_masks:
        print('No masks found.'); return

    pixel_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
    slice_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
    covid_n = luna_n = 0

    for mask_path in tqdm(all_masks, desc='Computing stats'):
        mask = np.load(mask_path)
        name = os.path.basename(mask_path)
        if name.startswith('covid_'): covid_n += 1
        elif name.startswith('luna_'): luna_n  += 1
        for cls in range(NUM_CLASSES):
            px = int(np.sum(mask == cls))
            pixel_counts[cls] += px
            if px > 0: slice_counts[cls] += 1

    total = pixel_counts.sum()
    names = ['Background', 'Lung', 'Nodule', 'Infection']

    print(f"\n{'='*62}")
    print(f" Total slices : {len(all_masks):,}   COVID: {covid_n:,}   LUNA: {luna_n:,}")
    print(f"{'='*62}")
    print(f"{'Class':<14} {'Pixels':>12} {'% total':>9} {'Slices w/ class':>18}")
    print(f"{'-'*56}")
    for cls in range(NUM_CLASSES):
        pct = 100.0 * pixel_counts[cls] / total if total > 0 else 0
        print(f"{names[cls]:<14} {pixel_counts[cls]:>12,} {pct:>8.2f}%  {slice_counts[cls]:>15,}")
    print(f"{'='*62}")

    # Median-frequency balancing (more stable than raw inverse frequency)
    # Prevents nodule weight from exploding when nodules are <0.5% of pixels
    freq          = pixel_counts.astype(np.float64) / total
    freq          = np.where(freq == 0, 1e-8, freq)
    median_freq   = np.median(freq)
    class_weights = np.clip(median_freq / freq, 0.1, 20.0)

    weight_str = ', '.join(f'{w:.4f}' for w in class_weights)
    print(f"\nSuggested class weights (median-frequency, capped at 20):")
    for cls in range(NUM_CLASSES):
        print(f"  Class {cls} ({names[cls]:<12}): {class_weights[cls]:.4f}")
    print(f"\nCopy into your training script:")
    print(f"  weights   = torch.tensor([{weight_str}], dtype=torch.float32).to(device)")
    print(f"  criterion = nn.CrossEntropyLoss(weight=weights)")


compute_dataset_stats()